# Workshop — 2. Emulate a Zero-Shot VLA Policy

Here we replace the scripted teacher from Notebook 1 with a small **Vision-Language-Action** model (~450M parameters):

```text
top + wrist images, joint state, instruction
                         ↓
                      SmolVLA
                         ↓
                  6 SO100 actions
```

**SmolVLA is the model.** `LerobotLocalPolicy` is the adapter that makes it a `Policy` compatible with `run_policy()`. A VLA therefore does not replace the policy abstraction: it implements its decision-making component.

> This is a local inference demonstration, not a guarantee of task success. The checkpoint was trained with real SO100 robots and cameras; the MuJoCo images are out of distribution.

## 1. Install and Configure the Runtime

The runtime and checkpoint are pinned to exact revisions. The first run downloads approximately 0.9 GB of model weights. Graphics and device environment variables must be set before importing MuJoCo or PyTorch.

In [ ]:
%pip install -q -r requirements.txt

### Configure graphics and device behavior

These variables select the platform-appropriate offscreen renderer and allow PyTorch to fall back to CPU for operations not implemented on Apple MPS. They must be set before importing MuJoCo or PyTorch.

In [ ]:
import os
import platform

os.environ.setdefault("MUJOCO_GL", "cgl" if platform.system() == "Darwin" else "egl")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")
if platform.system() == "Darwin":
    os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")

## 2. Define the Model and Robot Contract

The policy must know the exact action order, instruction, checkpoint revision, and units. The starting pose is the mean state of the checkpoint's original real-robot dataset. Its first five joints are stored in degrees and its gripper uses a 0–100 convention, while MuJoCo expects radians and the simulated jaw range.

In [ ]:
from pathlib import Path

import mujoco
import numpy as np
import torch
from IPython.display import Video, display
from PIL import Image
from strands_robots import Robot
from strands_robots.policies import create_policy

MODEL_ID = "lucarrr/smolvla_so100_pickplace_finetuned_v2"
MODEL_REVISION = "4fde42badae91b5c88bfe1a399a3f4b994fc0698"
INSTRUCTION = "Pick up the cube and place it in the box."
JOINT_KEYS = ["Rotation", "Pitch", "Elbow", "Wrist_Pitch", "Wrist_Roll", "Jaw"]
DATASET_MEAN_STATE = np.array(
    [14.4717, -55.7695, 54.3857, 63.2262, 85.8417, 9.3545],
    dtype=np.float64,
)
GRIPPER_JOINT_RANGE = (-0.175, 1.745)
BOX_CENTER = np.array([0.16, -0.30], dtype=np.float64)
VIDEO_PATH = Path("smolvla_pick.mp4").resolve()

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Model:", f"{MODEL_ID}@{MODEL_REVISION[:8]}")
print("Device:", device)
print("Instruction:", INSTRUCTION)

## 3. Build the MuJoCo Scene Step by Step

`Robot("so100")` already creates a safe MuJoCo world containing the robot. We then add the movable cube, construct the target as five static collision boxes, and attach two external cameras.

In [ ]:
def require_success(result, operation):
    """Raise when a strands-robots operation returns an error envelope."""
    if result.get("status") == "success":
        return result
    text = " | ".join(
        str(item.get("text"))
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("text")
    )
    raise RuntimeError(f"{operation} failed: {text or result}")


sim = Robot("so100", mesh=False)
require_success(
    sim.add_object(
        name="cube",
        shape="box",
        position=[0.0, -0.35, 0.015],
        size=[0.03, 0.03, 0.03],
        color=[0.9, 0.12, 0.08, 1.0],
        mass=0.03,
    ),
    "add cube",
)

blue = [0.12, 0.28, 0.75, 1.0]
x, y = BOX_CENTER
box_parts = {
    "target_base": ([x, y, 0.005], [0.12, 0.12, 0.01]),
    "target_left": ([x - 0.055, y, 0.025], [0.01, 0.12, 0.05]),
    "target_right": ([x + 0.055, y, 0.025], [0.01, 0.12, 0.05]),
    "target_back": ([x, y + 0.055, 0.025], [0.10, 0.01, 0.05]),
    "target_front": ([x, y - 0.055, 0.025], [0.10, 0.01, 0.05]),
}
for name, (position, size) in box_parts.items():
    require_success(
        sim.add_object(
            name=name,
            shape="box",
            position=list(position),
            size=list(size),
            color=blue,
            is_static=True,
        ),
        f"add {name}",
    )

### Add observations and restore the start pose

The two cameras provide the visual inputs expected by SmolVLA. The final action moves the simulated arm to the mean pose of the original training dataset, reducing an avoidable state-distribution mismatch.

In [ ]:
require_success(
    sim.add_camera(
        name="top",
        position=[0.45, -0.60, 0.42],
        target=[0.06, -0.31, 0.06],
        fov=55,
        width=640,
        height=480,
    ),
    "add top camera",
)
require_success(
    sim.add_camera(
        name="wrist",
        position=[0.02, -0.56, 0.18],
        target=[0.04, -0.31, 0.04],
        fov=70,
        width=640,
        height=480,
    ),
    "add wrist camera",
)

start_values = np.empty(6, dtype=np.float64)
start_values[:5] = np.deg2rad(DATASET_MEAN_STATE[:5])
jaw_min, jaw_max = GRIPPER_JOINT_RANGE
start_values[5] = jaw_min + (DATASET_MEAN_STATE[5] / 100.0) * (jaw_max - jaw_min)
start_action = dict(zip(JOINT_KEYS, start_values.tolist(), strict=True))
require_success(
    sim.send_action(start_action, robot_name="so100", n_substeps=600),
    "move SO100 to the dataset-mean pose",
)

observation = sim.get_observation("so100")
display(Image.fromarray(observation["top"]).resize((480, 360)))
display(Image.fromarray(observation["wrist"]).resize((480, 360)))

Inspect both images before continuing. Each camera must provide useful evidence about the cube and robot. This is an input-contract check, not merely visualization.

## 4. Define the Embodiment Adapter

The checkpoint expects image features named `camera1` and `camera2`, real-arm joint values in degrees, and a 0–100 gripper value. The simulation produces cameras named `wrist` and `top` and joint values in MuJoCo units. The embodiment adapter performs that translation while preserving the six-joint order.

In [ ]:
baseline_embodiment = {
    "name": "so100_smolvla_pickplace_sim",
    "obs_rename": {
        "wrist": "observation.images.camera1",
        "top": "observation.images.camera2",
    },
    "state_keys": JOINT_KEYS,
    "action_keys": JOINT_KEYS,
    "dim_policy": "strict",
    "state_units": "degrees",
    "action_units": "degrees",
    "gripper_index": 5,
    "gripper_joint_range": list(GRIPPER_JOINT_RANGE),
}

policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path=MODEL_ID,
    revision=MODEL_REVISION,
    policy_type="smolvla",
    device=device,
    embodiment=baseline_embodiment,
    strict_keys=True,
)

print("Policy adapter:", type(policy).__name__)
print("requires_images:", policy.requires_images)
print("execution_horizon:", policy.execution_horizon)

## 5. Run the Policy Loop

SmolVLA emits action chunks. `run_policy()` repeatedly collects images and joint state, requests an action chunk, applies those actions to MuJoCo, and records the `top` camera. We execute 400 control steps at 30 Hz, approximately the duration of the checkpoint's training demonstrations.

In [ ]:
result = sim.run_policy(
    robot_name="so100",
    policy_object=policy,
    instruction=INSTRUCTION,
    n_steps=400,
    control_frequency=30,
    fast_mode=True,
    video={
        "path": str(VIDEO_PATH),
        "camera": "top",
        "fps": 30,
    },
)
print("run_policy status:", result["status"])

## 6. Measure Physical Task Success

Software status and task success are different measurements. The cube succeeds only when its center is inside the target's horizontal bounds and below the top of the tray.

In [ ]:
cube_id = mujoco.mj_name2id(sim.mj_model, mujoco.mjtObj.mjOBJ_BODY, "cube")
cube_position = np.asarray(sim.mj_data.xpos[cube_id], dtype=float).copy()
inside_box_xy = bool(
    np.all(np.abs(cube_position[:2] - BOX_CENTER) < np.array([0.045, 0.045]))
)
diagnostics = {
    "cube_position_m": np.round(cube_position, 4).tolist(),
    "placed_in_box": inside_box_xy and cube_position[2] < 0.10,
}

print("Task diagnostics:", diagnostics)
display(Video(str(VIDEO_PATH), embed=True, width=640))

## What We Demonstrated

- The VLA is inside the policy: it uses pixels, language, and state to generate actions.
- The notebook explicitly constructed the scene, embodiment mapping, policy, rollout, and physical metric.
- `status="success"` means that the rollout executed without software errors; it does **not** mean that the task succeeded.
- If `placed_in_box=False`, the likely cause is the real-camera → MuJoCo domain shift. Notebook 3 fine-tunes SmolVLA on the matching demonstrations collected in Notebook 1.

The reusable equivalent remains available in `code/vla_pick.py`, but this notebook does not import it.